In [ ]:
# Step 1: Install required packages
# This keeps the original workflow and installs only the libraries needed for GEO microarray processing.
!pip install -q GEOparse pandas numpy matplotlib seaborn mygene

In [ ]:
# Step 2: Import necessary libraries
import os
import re
import warnings

import pandas as pd
import numpy as np
import GEOparse
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files

warnings.filterwarnings("ignore", category=FutureWarning)


# =============================================================================
# SMALL HELPER FUNCTIONS
# These functions only make the original cells compatible with different GPL
# annotation structures. They do not change the workflow.
# =============================================================================

def normalize_column_name(name):
    """Normalize a column name for safe platform-independent matching."""
    return re.sub(r"[^a-z0-9]+", "", str(name).lower())


def find_annotation_column(columns, exact_candidates=None, token_groups=None):
    """Find an annotation column across Affymetrix, Illumina, Agilent, or other GPL tables."""
    columns = list(columns)
    normalized = {col: normalize_column_name(col) for col in columns}

    for candidate in exact_candidates or []:
        candidate_norm = normalize_column_name(candidate)
        for col, col_norm in normalized.items():
            if col_norm == candidate_norm:
                return col

    for tokens in token_groups or []:
        token_norms = [normalize_column_name(token) for token in tokens]
        for col, col_norm in normalized.items():
            if all(token in col_norm for token in token_norms):
                return col

    return None


def flatten_metadata_value(value):
    """Convert GEO metadata lists into readable text without losing information."""
    if isinstance(value, (list, tuple)):
        cleaned = [str(item).strip() for item in value if str(item).strip()]
        return " | ".join(cleaned)
    if value is None:
        return np.nan
    return value


def get_gsm_platform_id(gsm):
    """Return the GPL accession used by one GSM sample."""
    value = gsm.metadata.get("platform_id", [])
    if isinstance(value, (list, tuple)):
        return str(value[0]).strip() if value else ""
    return str(value).strip()


def clean_gene_symbol(value):
    """Clean a direct gene-symbol annotation while retaining the original first-symbol rule."""
    if pd.isna(value):
        return np.nan

    text = str(value).strip()
    if not text or text.lower() in {"nan", "na", "n/a", "none", "null", "---", "-"}:
        return np.nan

    # The original workflow used the first symbol before ' /// '.
    text = re.split(r"\s*///\s*", text)[0].strip()

    # Some GPLs separate alternatives with semicolons or pipes.
    text = re.split(r"\s*[;|]\s*", text)[0].strip()

    if not text or " " in text:
        return np.nan

    if re.fullmatch(r"[A-Za-z][A-Za-z0-9._-]*", text):
        return text.upper()

    return np.nan


def extract_symbol_from_assignment(value):
    """Extract a likely human gene symbol from Affymetrix/Agilent assignment text."""
    if pd.isna(value):
        return np.nan

    text = str(value).strip()
    if not text:
        return np.nan

    tokens = [
        token.strip()
        for token in re.split(r"\s*///\s*|\s*//\s*|\s*[;|]\s*", text)
        if token.strip()
    ]

    accession_prefixes = (
        "NM_", "NR_", "XM_", "XR_", "ENSG", "ENST", "ENSP",
        "ILMN_", "A_", "AFFX", "GI:", "GB:", "REFSEQ:", "ENTREZ"
    )

    for token in tokens:
        token = token.strip("[](){}\"'")
        if token.upper().startswith(accession_prefixes):
            continue
        if re.fullmatch(r"[A-Za-z][A-Za-z0-9._-]{1,30}", token) and " " not in token:
            return token.upper()

    return np.nan


def first_identifier(value):
    """Return the first Entrez or RefSeq identifier from a multi-value annotation."""
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    if not text or text.lower() in {"nan", "na", "n/a", "none", "null", "---", "-"}:
        return np.nan
    return re.split(r"\s*///\s*|\s*//\s*|\s*[;,|]\s*", text)[0].strip()


def map_identifiers_with_mygene(values, scopes, species="human"):
    """Optional fallback when a GPL supplies Entrez/RefSeq IDs but no gene-symbol column."""
    identifiers = pd.Series(values).dropna().map(first_identifier).dropna().astype(str)
    identifiers = identifiers[identifiers.str.strip().ne("")].drop_duplicates().tolist()
    if not identifiers:
        return {}

    try:
        import mygene
        mg = mygene.MyGeneInfo()
        mapping = {}

        for start in range(0, len(identifiers), 1000):
            batch = identifiers[start:start + 1000]
            results = mg.querymany(
                batch,
                scopes=scopes,
                fields="symbol",
                species=species,
                as_dataframe=False,
                verbose=False,
            )
            for result in results:
                query = str(result.get("query", "")).strip()
                symbol = clean_gene_symbol(result.get("symbol"))
                if query and pd.notna(symbol):
                    mapping.setdefault(query, symbol)

        return mapping

    except Exception as error:
        print(f"MyGene fallback warning: {error}")
        return {}

In [ ]:
# Step 3: Download the selected GEO microarray dataset
# EDIT ONLY THIS CELL FOR A NEW DATASET.

GSE_ID = "GSE19420"          # Example: GSE19420, GSE27011, GSE66407
PLATFORM_ID = None            # None = automatically use the GPL containing the most samples
USE_MYGENE_FALLBACK = True    # Used only when GPL annotation lacks a usable gene-symbol column
DOWNLOAD_INTERMEDIATE_FILES = False  # Keep False to download only the final processed CSV

print(f"Step 3: Downloading {GSE_ID} dataset...")
gse = GEOparse.get_GEO(geo=GSE_ID, destdir="./", silent=False)
print("Dataset downloaded successfully!")
print("Available platforms:", list(gse.gpls.keys()))

In [ ]:
# Step 4: Extract the expression data
print("\nStep 4: Extracting expression data...")

# Keep the original idea of selecting one platform, but make the selection safe
# when a GEO series contains more than one GPL.
platform_sample_counts = {}
for gsm_name, gsm in gse.gsms.items():
    gsm_platform = get_gsm_platform_id(gsm)
    platform_sample_counts[gsm_platform] = platform_sample_counts.get(gsm_platform, 0) + 1

if PLATFORM_ID is None:
    valid_platform_counts = {
        key: value
        for key, value in platform_sample_counts.items()
        if key in gse.gpls
    }
    if valid_platform_counts:
        platform_id = max(valid_platform_counts, key=valid_platform_counts.get)
    else:
        platform_id = list(gse.gpls.keys())[0]
else:
    platform_id = str(PLATFORM_ID)

if platform_id not in gse.gpls:
    raise ValueError(
        f"Selected platform {platform_id} is not available. "
        f"Available platforms: {list(gse.gpls.keys())}"
    )

gpl = platform_id
print("Selected platform:", platform_id)
print("Samples per platform:", platform_sample_counts)

# Get expression data from the dataset
expression_set = pd.DataFrame()
processed_sample_ids = []
skipped_samples = []

# Process each sample exactly as in the original workflow
for gsm_name, gsm in gse.gsms.items():
    gsm_platform = get_gsm_platform_id(gsm)
    if gsm_platform and gsm_platform != platform_id:
        continue

    table = gsm.table.copy()

    id_ref_col = find_annotation_column(
        table.columns,
        exact_candidates=["ID_REF", "ID", "Probe ID", "ProbeID", "SPOT_ID"],
        token_groups=[["id", "ref"], ["probe", "id"]],
    )
    value_col = find_annotation_column(
        table.columns,
        exact_candidates=["VALUE", "Signal", "Normalized Signal", "Processed Signal"],
        token_groups=[["value"], ["signal"]],
    )

    if id_ref_col is None or value_col is None:
        skipped_samples.append({
            "sample_id": gsm_name,
            "reason": f"ID/value columns not detected; columns={table.columns.tolist()}"
        })
        continue

    sample_data = table[[id_ref_col, value_col]].copy()
    sample_data.columns = ["ID_REF", gsm_name]
    sample_data["ID_REF"] = sample_data["ID_REF"].astype(str).str.strip()
    sample_data[gsm_name] = pd.to_numeric(sample_data[gsm_name], errors="coerce")
    sample_data = sample_data.drop_duplicates(subset="ID_REF", keep="first")

    # If this is the first sample, use it as the base dataframe
    if expression_set.empty:
        expression_set = sample_data
    else:
        # Merge with existing data, preserving the original outer-merge logic
        expression_set = pd.merge(expression_set, sample_data, on="ID_REF", how="outer")

    processed_sample_ids.append(gsm_name)

if expression_set.empty:
    raise RuntimeError(
        "No expression data could be extracted. Inspect the GSM table columns and selected platform."
    )

# Rename ID_REF to ID for consistency with the original workflow
expression_set.rename(columns={"ID_REF": "ID"}, inplace=True)

print("Expression data shape:", expression_set.shape)
print("Processed samples:", len(processed_sample_ids))
print("Skipped samples:", len(skipped_samples))
if skipped_samples:
    display(pd.DataFrame(skipped_samples).head())

print("Expression data preview:")
display(expression_set.head())

In [ ]:
# @title Default title text
# FEATURE DATA FOR ILLUMINA (IF NaN VALUES ARE SEEN)
# ORIGINAL CUSTOM INSPECTION STEP RETAINED.
# It is now safe for Illumina, Affymetrix, Agilent, and other GPL annotation tables.

# Safety initialization fixes the original NameError without moving this cell.
if "platform_id" not in globals():
    platform_id = PLATFORM_ID if PLATFORM_ID is not None else list(gse.gpls.keys())[0]
if "feature_data" not in globals():
    feature_data = gse.gpls[platform_id].table.copy()

probe_id_preview_col = find_annotation_column(
    feature_data.columns,
    exact_candidates=["ID", "ID_REF", "Probe ID", "ProbeID", "Probe_Id", "SPOT_ID", "NAME"],
    token_groups=[["probe", "id"], ["spot", "id"]],
)
symbol_preview_col = find_annotation_column(
    feature_data.columns,
    exact_candidates=[
        "Symbol", "Gene Symbol", "Gene symbol", "GENE_SYMBOL", "GeneSymbol",
        "GENE", "ILMN_Gene", "Gene_Symbol", "SYMBOL"
    ],
    token_groups=[["gene", "symbol"]],
)
entrez_preview_col = find_annotation_column(
    feature_data.columns,
    exact_candidates=["Entrez_Gene_ID", "ENTREZ_GENE_ID", "Gene ID", "GENE_ID", "Entrez Gene"],
    token_groups=[["entrez"], ["gene", "id"]],
)
refseq_preview_col = find_annotation_column(
    feature_data.columns,
    exact_candidates=["RefSeq_ID", "REFSEQ", "RefSeq", "RefSeq Transcript ID"],
    token_groups=[["refseq"]],
)
probe_type_preview_col = find_annotation_column(
    feature_data.columns,
    exact_candidates=["Probe_Type", "CONTROL_TYPE", "Control Type", "SPOT_TYPE"],
    token_groups=[["probe", "type"], ["control", "type"]],
)
chromosome_preview_col = find_annotation_column(
    feature_data.columns,
    exact_candidates=["Chromosome", "CHR", "Chromosomal Location"],
    token_groups=[["chromosome"], ["chromosomal", "location"]],
)
definition_preview_col = find_annotation_column(
    feature_data.columns,
    exact_candidates=["Definition", "Gene title", "Gene Title", "GENE_NAME", "Description"],
    token_groups=[["gene", "title"], ["description"]],
)

# Keep biological probes when at least one useful biological annotation is present.
annotation_presence = pd.Series(False, index=feature_data.index)
for selected_col in [symbol_preview_col, entrez_preview_col, refseq_preview_col]:
    if selected_col is not None:
        valid = feature_data[selected_col].notna() & feature_data[selected_col].astype(str).str.strip().ne("")
        annotation_presence = annotation_presence | valid

# If the platform has no standard biological-annotation columns, retain all rows for inspection.
if not annotation_presence.any():
    annotation_presence = pd.Series(True, index=feature_data.index)

# Exclude obvious technical controls when the GPL supplies a control/probe-type column.
if probe_type_preview_col is not None:
    technical_control = feature_data[probe_type_preview_col].astype(str).str.contains(
        r"control|spike|blank|negative|positive|housekeeping", case=False, regex=True, na=False
    )
else:
    technical_control = pd.Series(False, index=feature_data.index)

biological_probes = feature_data.loc[annotation_presence & ~technical_control].copy()

# Keep the original variable name 'illumina_features' so the user's workflow remains familiar.
preview_columns = [
    probe_id_preview_col,
    symbol_preview_col,
    entrez_preview_col,
    refseq_preview_col,
    probe_type_preview_col,
    chromosome_preview_col,
    definition_preview_col,
]
preview_columns = [col for col in preview_columns if col is not None]
preview_columns = list(dict.fromkeys(preview_columns))

illumina_features = (
    biological_probes[preview_columns].copy()
    if preview_columns
    else biological_probes.copy()
)

print("Filtered Microarray Features (Biological Probes Only):")
display(illumina_features.head())
print("\nShape after filtering:", illumina_features.shape)
print("Detected preview columns:", preview_columns)

In [ ]:
# @title Default title text
# Step 5: Get feature data (probe annotations)
print("\nStep 5: Getting feature data...")

# Re-read the same selected GPL, retaining the original Step 5 inspection cell.
if "platform_id" not in globals():
    platform_id = PLATFORM_ID if PLATFORM_ID is not None else list(gse.gpls.keys())[0]

feature_data = gse.gpls[platform_id].table.copy()

# Display the feature data columns to verify structure
print("Platform ID:", platform_id)
print("Feature data shape:", feature_data.shape)
print("Feature data columns:", feature_data.columns.tolist())
print("Feature data preview:")
display(feature_data.head())

In [ ]:
# Step 6: Check for correct column names for gene symbol
print("\nStep 6: Checking column names for gene symbol...")

# Editable candidate lists are intentionally visible so this step can still be customized.
possible_probe_id_cols = [
    "ID", "ID_REF", "Probe ID", "ProbeID", "Probe_Id", "PROBE_ID",
    "SPOT_ID", "NAME", "Reporter Identifier", "Array_Address_Id"
]

possible_gene_symbol_cols = [
    "gene_symbol", "Gene symbol", "Gene Symbol", "Symbol", "SYMBOL",
    "gene symbol", "GENE_SYMBOL", "GeneSymbol", "Gene_Symbol",
    "ILMN_Gene", "GENE"
]

possible_entrez_cols = [
    "Entrez_Gene_ID", "ENTREZ_GENE_ID", "Entrez Gene", "EntrezGene ID",
    "Gene ID", "GENE_ID", "ENTREZID"
]

possible_refseq_cols = [
    "RefSeq_ID", "REFSEQ", "RefSeq", "RefSeq Transcript ID", "RefSeq_IDs"
]

possible_assignment_cols = [
    "gene_assignment", "Gene Assignment", "Gene assignment",
    "Target Description", "TARGET_DESCRIPTION", "Description"
]

possible_control_cols = [
    "Probe_Type", "CONTROL_TYPE", "Control Type", "SPOT_TYPE", "ControlType"
]

probe_id_col = find_annotation_column(
    feature_data.columns,
    exact_candidates=possible_probe_id_cols,
    token_groups=[["probe", "id"], ["spot", "id"], ["reporter", "identifier"]],
)

gene_symbol_col = find_annotation_column(
    feature_data.columns,
    exact_candidates=possible_gene_symbol_cols,
    token_groups=[["gene", "symbol"]],
)

entrez_gene_col = find_annotation_column(
    feature_data.columns,
    exact_candidates=possible_entrez_cols,
    token_groups=[["entrez"], ["gene", "id"]],
)

refseq_col = find_annotation_column(
    feature_data.columns,
    exact_candidates=possible_refseq_cols,
    token_groups=[["refseq"]],
)

gene_assignment_col = find_annotation_column(
    feature_data.columns,
    exact_candidates=possible_assignment_cols,
    token_groups=[["gene", "assignment"], ["target", "description"]],
)

control_probe_col = find_annotation_column(
    feature_data.columns,
    exact_candidates=possible_control_cols,
    token_groups=[["control", "type"], ["probe", "type"]],
)

if probe_id_col is None:
    raise ValueError(
        "Could not identify the probe-ID column. Available columns are: "
        + str(feature_data.columns.tolist())
    )

print("Detected probe ID column       :", probe_id_col)
print("Detected gene symbol column    :", gene_symbol_col)
print("Detected Entrez ID column      :", entrez_gene_col)
print("Detected RefSeq column         :", refseq_col)
print("Detected assignment column     :", gene_assignment_col)
print("Detected control-probe column  :", control_probe_col)

if gene_symbol_col is None:
    print(
        "No direct gene-symbol column was found. Step 6 will use assignment text, "
        "Entrez ID, or RefSeq ID as a controlled fallback."
    )

In [ ]:
# Step 6: Create gene_info dataframe
print("\nStep 6: Creating gene information table...")

# Retain the original gene_info structure: probe_id, gene_symbol, gene_id.
annotation_source = feature_data.copy()

gene_info = pd.DataFrame({
    "probe_id": annotation_source[probe_id_col].astype(str).str.strip(),
    "gene_symbol": (
        annotation_source[gene_symbol_col]
        if gene_symbol_col is not None
        else pd.Series(pd.NA, index=annotation_source.index, dtype="object")
    ),
    "gene_id": annotation_source[probe_id_col].astype(str).str.strip(),
})

# Keep optional source columns temporarily for fallback mapping.
if entrez_gene_col is not None:
    gene_info["_entrez"] = annotation_source[entrez_gene_col]
else:
    gene_info["_entrez"] = np.nan

if refseq_col is not None:
    gene_info["_refseq"] = annotation_source[refseq_col]
else:
    gene_info["_refseq"] = np.nan

if gene_assignment_col is not None:
    gene_info["_assignment"] = annotation_source[gene_assignment_col]
else:
    gene_info["_assignment"] = np.nan

# Clean direct symbols using the same first-symbol principle as the original code.
gene_info["gene_symbol"] = gene_info["gene_symbol"].map(clean_gene_symbol)

# Affymetrix and some Agilent GPLs store symbols inside assignment text.
missing_symbol = gene_info["gene_symbol"].isna()
if missing_symbol.any() and gene_assignment_col is not None:
    gene_info.loc[missing_symbol, "gene_symbol"] = (
        gene_info.loc[missing_symbol, "_assignment"].map(extract_symbol_from_assignment)
    )

# Optional fallback for GPLs that contain Entrez IDs but no usable symbol column.
missing_symbol = gene_info["gene_symbol"].isna()
if USE_MYGENE_FALLBACK and missing_symbol.any() and entrez_gene_col is not None:
    print("Mapping unresolved Entrez IDs to gene symbols using MyGene...")
    entrez_map = map_identifiers_with_mygene(
        gene_info.loc[missing_symbol, "_entrez"],
        scopes="entrezgene",
        species="human",
    )
    first_entrez = gene_info["_entrez"].map(first_identifier)
    gene_info.loc[missing_symbol, "gene_symbol"] = first_entrez.loc[missing_symbol].map(entrez_map)

# Optional RefSeq fallback if required.
missing_symbol = gene_info["gene_symbol"].isna()
if USE_MYGENE_FALLBACK and missing_symbol.any() and refseq_col is not None:
    print("Mapping unresolved RefSeq IDs to gene symbols using MyGene...")
    refseq_map = map_identifiers_with_mygene(
        gene_info.loc[missing_symbol, "_refseq"],
        scopes="refseq",
        species="human",
    )
    first_refseq = gene_info["_refseq"].map(first_identifier)
    gene_info.loc[missing_symbol, "gene_symbol"] = first_refseq.loc[missing_symbol].map(refseq_map)

# Exclude obvious technical controls only when the GPL provides an explicit control column.
if control_probe_col is not None:
    technical_control = annotation_source[control_probe_col].astype(str).str.contains(
        r"control|spike|blank|negative|positive|housekeeping",
        case=False,
        regex=True,
        na=False,
    )
    gene_info = gene_info.loc[~technical_control].copy()

# Remove rows with missing gene symbols or probe IDs.
gene_info["gene_symbol"] = gene_info["gene_symbol"].map(clean_gene_symbol)
gene_info = gene_info[
    gene_info["gene_symbol"].notna()
    & gene_info["gene_symbol"].astype(str).str.strip().ne("")
    & gene_info["probe_id"].notna()
    & gene_info["probe_id"].astype(str).str.strip().ne("")
].copy()

gene_info = gene_info.drop_duplicates(subset="probe_id", keep="first")
gene_info = gene_info[["probe_id", "gene_symbol", "gene_id"]]

print("Gene info shape:", gene_info.shape)
print("Gene info preview:")
display(gene_info.head())

In [ ]:
# Create phenotype dataframe with full metadata
# ORIGINAL PHENOTYPE INSPECTION STEP RETAINED.

phenotype_rows = []
sample_ids_for_metadata = processed_sample_ids if "processed_sample_ids" in globals() else list(gse.gsms.keys())

for sample_id in sample_ids_for_metadata:
    gsm = gse.gsms[sample_id]
    metadata = {key: flatten_metadata_value(value) for key, value in gsm.metadata.items()}

    characteristics = gsm.metadata.get("characteristics_ch1", [])
    if not isinstance(characteristics, (list, tuple)):
        characteristics = [characteristics]

    row = {
        "sample_id": gsm.name,
        **metadata,
        **{
            f"characteristics_{i}": flatten_metadata_value(char)
            for i, char in enumerate(characteristics)
        },
    }
    phenotype_rows.append(row)

pheno_data = pd.DataFrame(phenotype_rows)
pheno_data.reset_index(drop=True, inplace=True)

print("Phenotype data shape:", pheno_data.shape)
print("Phenotype data columns:")
print(pheno_data.columns.tolist())
print("Phenotype data preview:")
display(pheno_data.head())

# Save the dataframe exactly as in the original workflow.
pheno_data.to_csv("pheno_data.csv", index=False)

# Keep False when only the final processed CSV should be downloaded.
if DOWNLOAD_INTERMEDIATE_FILES:
    files.download("pheno_data.csv")

In [ ]:
print("\nStep 7: Processing sample information from `pheno_data`...")

# EDIT THIS VALUE after checking the phenotype table or the unique-value cell below.
description_column = "characteristics_1"

# Preserve the original preference for characteristics_1, but prevent failure when
# another GEO dataset uses a different characteristics column.
if description_column not in pheno_data.columns or pheno_data[description_column].dropna().empty:
    characteristics_candidates = [
        col for col in pheno_data.columns
        if str(col).startswith("characteristics_") and pheno_data[col].notna().any()
    ]

    fallback_candidates = characteristics_candidates + [
        col for col in ["title", "source_name_ch1", "description"]
        if col in pheno_data.columns and pheno_data[col].notna().any()
    ]

    if not fallback_candidates:
        raise ValueError(
            "No usable sample-description column was found. Available columns: "
            + str(pheno_data.columns.tolist())
        )

    selected_description_column = fallback_candidates[0]
    print(
        f"Requested column '{description_column}' was unavailable. "
        f"Automatically using '{selected_description_column}'."
    )
else:
    selected_description_column = description_column

sample_info = pd.DataFrame({
    "geo_accession": pheno_data["sample_id"].astype(str),
    "description": pheno_data[selected_description_column].map(flatten_metadata_value),
})

print("Selected description column:", selected_description_column)
print("Sample info preview:")
display(sample_info.head())

In [ ]:
# Step 8: Create a mapping between GSM IDs and descriptions
print("\nStep 8: Creating sample description mapping...")

# Create a mapping dictionary while retaining GSM IDs when a description is missing.
desc_mapping = {}
for gsm_id, description in zip(sample_info["geo_accession"], sample_info["description"]):
    if pd.isna(description) or not str(description).strip():
        desc_mapping[str(gsm_id)] = str(gsm_id)
    else:
        desc_mapping[str(gsm_id)] = str(description).strip()

print("Description mapping (first 5 items):")
for i, (key, value) in enumerate(desc_mapping.items()):
    print(f"{key}: {value}")
    if i >= 4:
        break

In [ ]:
# Step 9: Replace GSM IDs with descriptions in expression_set
print("\nStep 9: Replacing GSM IDs with descriptions...")

original_cols = expression_set.columns.tolist()
new_cols = []

for col in original_cols:
    if col == "ID":
        new_cols.append(col)
    else:
        # Preserve the original behavior: repeated biological descriptions may create
        # repeated column names, which are later renamed manually to control/disease.
        new_cols.append(desc_mapping.get(str(col), col))

# Assign the complete column list directly so every repeated description is retained.
expression_set.columns = new_cols

print("Updated column names (first 20):")
print(expression_set.columns.tolist()[:20])

# Save the DataFrame as in the original workflow.
expression_set.to_csv("expression_set.csv", index=False)

if DOWNLOAD_INTERMEDIATE_FILES:
    files.download("expression_set.csv")

In [ ]:
# prompt: check the chosen column and give me every unique name in list
# ORIGINAL MANUAL CUSTOMIZATION STEP RETAINED.

# Replace this with the metadata column you want to inspect.
column_name = "characteristics_1"

if column_name in pheno_data.columns:
    unique_names = (
        pheno_data[column_name]
        .dropna()
        .map(flatten_metadata_value)
        .astype(str)
        .drop_duplicates()
        .tolist()
    )
    print(f"Unique values in column '{column_name}':")
    for value in unique_names:
        print(value)
else:
    print(f"Column '{column_name}' not found in the dataframe.")
    print("Available columns are:", pheno_data.columns.tolist())

In [ ]:
# ORIGINAL MANUAL EXCLUSION STEP RETAINED.
# Add or remove exact sample-description labels after inspecting the previous cell.
columns_to_drop = [
    "disease status: mild/moderate asthma",
]

existing_columns_to_drop = [
    col for col in columns_to_drop
    if col in expression_set.columns
]

if existing_columns_to_drop:
    expression_set.drop(columns=existing_columns_to_drop, inplace=True)
    print("Dropped columns:", existing_columns_to_drop)
else:
    print("None of the requested exclusion labels were present. No columns were dropped.")

expression_set.to_csv("expression_set_updated.csv", index=False)

if DOWNLOAD_INTERMEDIATE_FILES:
    files.download("expression_set_updated.csv")

In [ ]:
# ORIGINAL MANUAL GROUP-RENAME STEP RETAINED.
# Edit the left side using the exact unique phenotype values printed earlier.
rename_mapping = {
    "disease status: severe asthma": "disease",
    "disease status: healthy": "control",
}

expression_set.rename(columns=rename_mapping, inplace=True)

print("All remaining columns:")
print(expression_set.columns.tolist())
print("Number of control columns:", sum(str(col) == "control" for col in expression_set.columns))
print("Number of disease columns:", sum(str(col) == "disease" for col in expression_set.columns))

expression_set.to_csv("expression_set_renamed.csv", index=False)

if DOWNLOAD_INTERMEDIATE_FILES:
    files.download("expression_set_renamed.csv")

In [ ]:
# Step 13: Merge expression data with gene information
print("\nStep 13: Merging datasets...")

# Standardize probe-ID type only; the original left merge is retained.
expression_set["ID"] = expression_set["ID"].astype(str).str.strip()
gene_info["probe_id"] = gene_info["probe_id"].astype(str).str.strip()

final_data = pd.merge(
    expression_set,
    gene_info,
    left_on="ID",
    right_on="probe_id",
    how="left",
)

print("Merged final_data shape:", final_data.shape)
display(final_data.head())

# Save merged data
final_data.to_csv("final_data.csv", index=False)

if DOWNLOAD_INTERMEDIATE_FILES:
    files.download("final_data.csv")

In [ ]:
# prompt: final_data.csv contains ID, probe_id, gene_symbol, and gene_id.
# Replace ID with gene_symbol and delete the specified probe columns.

import pandas as pd
from google.colab import files

# Load the final_data.csv file
final_data = pd.read_csv("final_data.csv")

if "gene_symbol" not in final_data.columns:
    raise KeyError("gene_symbol was not created during probe annotation.")

# Retain the original replacement operation.
final_data["ID"] = final_data["gene_symbol"]

# Drop the unnecessary columns safely.
final_data.drop(columns=["probe_id", "gene_id"], inplace=True, errors="ignore")

# Save the modified DataFrame to a new CSV file
final_data.to_csv("modified_final_data.csv", index=False)

if DOWNLOAD_INTERMEDIATE_FILES:
    files.download("modified_final_data.csv")

In [ ]:
# @title Default title text
# prompt: analyze modified_final_data.csv and count control and disease columns

import pandas as pd

modified_final_data = pd.read_csv("modified_final_data.csv")

print(modified_final_data.head())

# CSV readers make repeated names unique as control, control.1, control.2, etc.
control_columns = [
    col for col in modified_final_data.columns
    if str(col) == "control" or str(col).startswith("control.")
]

disease_columns = [
    col for col in modified_final_data.columns
    if str(col) in {"disease", "case"}
    or str(col).startswith("disease.")
    or str(col).startswith("case.")
]

print("\nNumber of control sample columns:", len(control_columns))
print("Control columns:", control_columns)
print("\nNumber of disease sample columns:", len(disease_columns))
print("Disease columns:", disease_columns)

if control_columns:
    print("\nControl expression summary:")
    print(modified_final_data[control_columns].apply(pd.to_numeric, errors="coerce").stack().describe())

if disease_columns:
    print("\nDisease expression summary:")
    print(modified_final_data[disease_columns].apply(pd.to_numeric, errors="coerce").stack().describe())

In [ ]:
import pandas as pd

# Load the modified CSV file
df = pd.read_csv("modified_final_data.csv")

# Keep a copy exactly as in the original workflow.
df_modified = df.copy()

# Remove accidental index columns while retaining the biological data.
unnamed_columns = [col for col in df_modified.columns if str(col).startswith("Unnamed:")]
if unnamed_columns:
    df_modified.drop(columns=unnamed_columns, inplace=True)

if "gene_symbol" not in df_modified.columns:
    if "ID" in df_modified.columns:
        df_modified.rename(columns={"ID": "gene_symbol"}, inplace=True)
    else:
        raise KeyError("Neither gene_symbol nor ID was found in modified_final_data.csv")

# Create the final DataFrame with gene_symbol as the first column and remove ID.
columns = ["gene_symbol"] + [
    col for col in df_modified.columns
    if col not in {"gene_symbol", "ID"}
]
df_final = df_modified[columns]

print(df_final.head())
print("Final reordered shape:", df_final.shape)

# Save to the same intermediate filename as the original workflow.
df_final.to_csv("modified_final_data.csv", index=False)

In [ ]:
# Load the modified CSV file
df = pd.read_csv("modified_final_data.csv")

# Drop rows where gene_symbol is blank or NaN, retaining the original final-cleaning rule.
df = df[
    df["gene_symbol"].notna()
    & df["gene_symbol"].astype(str).str.strip().ne("")
].copy()

# Dynamic filename prevents the dataset-name mismatch present in the original notebook.
final_output_filename = f"cleaned_modified_final_data {GSE_ID}.csv"

df.to_csv(final_output_filename, index=False)

print("Final processed file:", final_output_filename)
print("Final processed shape:", df.shape)
print("Gene-symbol column is first:", df.columns[0] == "gene_symbol")

# This is the only automatic download when DOWNLOAD_INTERMEDIATE_FILES=False.
files.download(final_output_filename)